# Customer Churn Prediction - SageMaker Training & Deployment

**Author**: Vahant Sharma  
**Date**: February 2026

## What This Notebook Does

1. **Connects to your S3 bucket** where data is already uploaded
2. **Prepares data** in the format XGBoost expects
3. **Launches a training job** on AWS infrastructure
4. **Deploys the model** as a real-time endpoint
5. **Tests predictions** on sample customers

## Cost Estimate

| Resource | Cost | Duration |
|----------|------|----------|
| Training (ml.m5.large) | ~$0.03 | 15 min |
| Endpoint (ml.t2.medium) | ~$0.05/hr | Until you delete |

**IMPORTANT**: Delete the endpoint when done testing to stop billing!

---
## Cell 1: Configuration

These are YOUR specific values. They're already filled in from your AWS setup.

In [ ]:
# ============================================================
# YOUR CONFIGURATION - These are YOUR specific AWS values
# ============================================================

# Your S3 bucket (you created this and uploaded data)
BUCKET_NAME = 'customer-churn-vahant-2026'

# Your IAM Role ARN (from AWS console - you saw this in IAM > Roles)
ROLE_ARN = 'arn:aws:iam::231284356634:role/service-role/AmazonSageMaker-ExecutionRole-20260212T230810'

# AWS Region
REGION = 'us-east-1'

# Print to verify
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"S3 Bucket:  {BUCKET_NAME}")
print(f"IAM Role:   {ROLE_ARN[:50]}...")
print(f"Region:     {REGION}")
print("=" * 60)

---
## Cell 2: Import Libraries

**What these libraries do:**
- `sagemaker`: AWS SDK for ML operations
- `boto3`: AWS SDK for general AWS services (S3, etc.)
- `pandas`: Data manipulation library

In [ ]:
# Standard imports
import sagemaker
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
import boto3
import pandas as pd
import numpy as np
import os
import json

# Initialize SageMaker session
# A session manages connections to SageMaker APIs
sess = sagemaker.Session()

# Print versions to confirm everything is working
print(f"SageMaker SDK version: {sagemaker.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Session region: {sess.boto_region_name}")
print("\n✅ All libraries imported successfully!")

---
## Cell 3: Verify Data in S3

**Why this step?**  
Before training, we need to confirm your data file exists in S3.  
You uploaded it earlier using: `aws s3 cp data/customer_churn_processed.csv s3://...`

In [ ]:
# Create S3 client to interact with S3
s3_client = boto3.client('s3', region_name=REGION)

# List files in your bucket's data/ folder
print(f"Checking S3 bucket: {BUCKET_NAME}")
print("-" * 50)

try:
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix='data/'
    )
    
    if 'Contents' in response:
        print("Files found in S3:")
        for obj in response['Contents']:
            size_mb = obj['Size'] / (1024 * 1024)
            print(f"  📁 {obj['Key']} ({size_mb:.2f} MB)")
        print("\n✅ Data is ready in S3!")
    else:
        print("❌ No files found in data/ folder!")
        print("\nPlease upload your data first using:")
        print(f"  aws s3 cp data/customer_churn_processed.csv s3://{BUCKET_NAME}/data/")
        
except Exception as e:
    print(f"❌ Error accessing S3: {e}")
    print("\nCheck that:")
    print("  1. Bucket name is correct")
    print("  2. You have S3 access permissions")

---
## Cell 4: Download and Explore Data

**Why this step?**  
We need to understand the data structure to:
1. Identify which column is the target (Churn)
2. See the features (input variables)
3. Check data types

In [ ]:
# Download data from S3 to local (temporary) storage
local_data_path = '/tmp/customer_churn.csv'
s3_data_key = 'data/customer_churn_processed.csv'

print(f"Downloading s3://{BUCKET_NAME}/{s3_data_key}...")
s3_client.download_file(BUCKET_NAME, s3_data_key, local_data_path)
print("✅ Download complete!\n")

# Load into pandas DataFrame
df = pd.read_csv(local_data_path)

# Show basic info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    print(f"  {i:2}. {col:<30} ({dtype})")

print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Show first few rows to understand the data
print("First 5 rows of data:")
df.head()

In [ ]:
# Check for target column (Churn)
print("Looking for target column...")

# Find columns containing 'churn' (case-insensitive)
churn_cols = [c for c in df.columns if 'churn' in c.lower()]

if churn_cols:
    TARGET_COLUMN = churn_cols[0]
    print(f"\n✅ Found target column: '{TARGET_COLUMN}'")
    print(f"\nTarget distribution:")
    print(df[TARGET_COLUMN].value_counts())
    print(f"\nChurn rate: {df[TARGET_COLUMN].mean() * 100:.2f}%")
else:
    print("❌ No 'Churn' column found!")
    print("Available columns:", list(df.columns))

---
## Cell 5: Prepare Data for XGBoost

**Why this step?**

XGBoost in SageMaker has specific requirements:
1. **Target column must be FIRST** - The thing we're predicting (Churn) goes in column 0
2. **No headers** - Training CSV files shouldn't have column names
3. **All numeric** - Convert categorical text to numbers

**What we're doing:**
- Moving Churn column to first position
- Converting Yes/No, Male/Female to 0/1
- Splitting into 80% training, 20% validation

In [ ]:
# Make a copy to avoid modifying original
df_prepared = df.copy()

# Step 1: Move target column to first position
print("Step 1: Reorganizing columns (target first)...")
cols = [TARGET_COLUMN] + [c for c in df_prepared.columns if c != TARGET_COLUMN]
df_prepared = df_prepared[cols]
print(f"  First column is now: '{df_prepared.columns[0]}'")

# Step 2: Convert categorical columns to numeric
print("\nStep 2: Converting categorical columns to numeric...")
categorical_mappings = {}

for col in df_prepared.columns:
    if df_prepared[col].dtype == 'object':
        # Store the mapping for reference
        unique_values = df_prepared[col].unique()
        
        # Convert to numeric codes
        df_prepared[col], mapping = pd.factorize(df_prepared[col])
        categorical_mappings[col] = dict(enumerate(mapping))
        
        print(f"  {col}: {list(unique_values)[:5]}... → numeric codes")

# Step 3: Verify all columns are numeric
print("\nStep 3: Verifying all columns are numeric...")
non_numeric = df_prepared.select_dtypes(exclude=[np.number]).columns
if len(non_numeric) == 0:
    print("  ✅ All columns are now numeric!")
else:
    print(f"  ❌ Non-numeric columns remaining: {list(non_numeric)}")

print(f"\nFinal shape: {df_prepared.shape}")

In [ ]:
# Show categorical mappings (useful for understanding predictions later)
print("Categorical Mappings (for reference):")
print("=" * 50)
for col, mapping in list(categorical_mappings.items())[:5]:
    print(f"\n{col}:")
    for code, value in mapping.items():
        print(f"  {code} = {value}")

In [ ]:
# Step 4: Split into training and validation sets
print("Step 4: Splitting data...")
print("  80% for training (model learns from this)")
print("  20% for validation (model tested on this)")

# Shuffle the data first (randomize order)
df_shuffled = df_prepared.sample(frac=1, random_state=42).reset_index(drop=True)

# Calculate split point
train_size = int(0.8 * len(df_shuffled))

# Split
train_df = df_shuffled[:train_size]
val_df = df_shuffled[train_size:]

print(f"\n  Training samples:   {len(train_df):,}")
print(f"  Validation samples: {len(val_df):,}")

# Step 5: Save to CSV (no headers, no index)
print("\nStep 5: Saving to CSV files...")

train_path = '/tmp/train.csv'
val_path = '/tmp/validation.csv'

train_df.to_csv(train_path, index=False, header=False)
val_df.to_csv(val_path, index=False, header=False)

print(f"  ✅ Saved: {train_path}")
print(f"  ✅ Saved: {val_path}")

---
## Cell 6: Upload Prepared Data to S3

**Why this step?**  
SageMaker training jobs run on separate instances (computers) in AWS.  
They can't access files on this notebook instance.  
S3 is the shared storage that both can access.

In [ ]:
# Upload train and validation data to S3
print("Uploading prepared data to S3...")
print("-" * 50)

# Use SageMaker's upload_data function (handles multipart upload for large files)
train_s3_uri = sess.upload_data(
    path=train_path,
    bucket=BUCKET_NAME,
    key_prefix='sagemaker/training-data'
)
print(f"✅ Training data:   {train_s3_uri}")

val_s3_uri = sess.upload_data(
    path=val_path,
    bucket=BUCKET_NAME,
    key_prefix='sagemaker/training-data'
)
print(f"✅ Validation data: {val_s3_uri}")

print("\n💡 These URIs will be passed to the training job.")

---
## Cell 7: Get XGBoost Container

**What is a Container?**  
A container is like a pre-packaged computer environment.  
AWS provides containers with XGBoost pre-installed.  
We just need to tell SageMaker which container version to use.

**Why XGBoost?**  
- Very fast training
- Handles tabular data well
- Good performance on churn prediction
- Built-in AWS support

In [ ]:
from sagemaker import image_uris

# Get the XGBoost container URI for our region
# Version 1.5-1 is stable and well-tested
xgboost_container = image_uris.retrieve(
    framework='xgboost',
    region=REGION,
    version='1.5-1'
)

print("XGBoost Container Information")
print("=" * 60)
print(f"Container URI: {xgboost_container}")
print(f"\nThis container includes:")
print("  - XGBoost 1.5.1")
print("  - Python 3.8")
print("  - All necessary dependencies")
print("\n✅ Container ready to use!")

---
## Cell 8: Create Training Estimator

**What is an Estimator?**  
An Estimator is a SageMaker object that defines HOW to train your model:
- What algorithm (XGBoost container)
- What hardware (instance type)
- Where to save output (S3 path)

**Instance Type Explanation:**
- `ml.m5.large` = 2 vCPUs, 8 GB RAM
- Cost: ~$0.115/hour
- Good balance of cost and speed

In [ ]:
# Create the XGBoost estimator
xgb_estimator = Estimator(
    
    # What container to use (XGBoost)
    image_uri=xgboost_container,
    
    # IAM role that SageMaker uses to access S3, CloudWatch, etc.
    role=ROLE_ARN,
    
    # Hardware configuration
    instance_count=1,              # Number of machines
    instance_type='ml.m5.large',   # Machine type (2 vCPU, 8GB RAM)
    
    # Where to save the trained model
    output_path=f's3://{BUCKET_NAME}/sagemaker/models/',
    
    # Session object
    sagemaker_session=sess,
    
    # Base name for the training job
    base_job_name='churn-xgboost',
    
    # Enable network isolation for security
    enable_network_isolation=False
)

print("Estimator Created!")
print("=" * 60)
print(f"Algorithm:     XGBoost 1.5")
print(f"Instance:      ml.m5.large (2 vCPU, 8GB RAM)")
print(f"Cost:          ~$0.115/hour")
print(f"Output:        s3://{BUCKET_NAME}/sagemaker/models/")

---
## Cell 9: Set Hyperparameters

**What are Hyperparameters?**  
Settings that control how the algorithm learns:

| Parameter | What it Does | Our Value |
|-----------|--------------|----------|
| objective | Type of problem | binary:logistic (yes/no prediction) |
| num_round | Training iterations | 100 |
| max_depth | Tree complexity | 5 (prevent overfitting) |
| eta | Learning rate | 0.2 (how fast to learn) |
| subsample | Row sampling | 0.8 (use 80% of data per tree) |
| eval_metric | How to measure success | auc (area under curve) |

In [ ]:
# Set hyperparameters
# These control how XGBoost learns patterns from your data

xgb_estimator.set_hyperparameters(
    
    # Objective: Binary classification (churn: yes or no)
    objective='binary:logistic',
    
    # Number of boosting rounds (iterations)
    # More rounds = more learning, but slower and risk of overfitting
    num_round=100,
    
    # Maximum depth of each tree
    # Deeper = more complex patterns, but risk of overfitting
    max_depth=5,
    
    # Learning rate (step size)
    # Lower = slower learning but more stable
    eta=0.2,
    
    # Subsample: fraction of data used per tree
    # Less than 1.0 reduces overfitting
    subsample=0.8,
    
    # Column sampling per tree
    colsample_bytree=0.8,
    
    # Evaluation metric: AUC is great for imbalanced classification
    eval_metric='auc'
)

print("Hyperparameters Set!")
print("=" * 60)
print("objective:        binary:logistic")
print("num_round:        100 boosting iterations")
print("max_depth:        5 (tree depth)")
print("eta:              0.2 (learning rate)")
print("subsample:        0.8 (80% row sampling)")
print("colsample_bytree: 0.8 (80% column sampling)")
print("eval_metric:      auc (area under curve)")

---
## Cell 10: Define Training Inputs

**What is a TrainingInput?**  
Tells SageMaker WHERE to find the data and HOW to read it.

- `s3_data`: S3 path to data
- `content_type`: File format (CSV)

In [ ]:
# Create TrainingInput objects
# These tell SageMaker where to find your data

train_input = TrainingInput(
    s3_data=train_s3_uri,
    content_type='text/csv'
)

val_input = TrainingInput(
    s3_data=val_s3_uri,
    content_type='text/csv'
)

print("Training Inputs Configured!")
print("=" * 60)
print(f"Training data:   {train_s3_uri}")
print(f"Validation data: {val_s3_uri}")
print(f"Content type:    text/csv")

---
## Cell 11: 🚀 START TRAINING

**THIS IS THE MAIN EVENT!**

When you run this cell:
1. SageMaker spins up an ml.m5.large EC2 instance
2. Downloads XGBoost container
3. Downloads your data from S3
4. Trains the model (100 rounds of boosting)
5. Uploads model.tar.gz to S3
6. Shuts down the instance

**Cost:** ~$0.03 for 15 minutes of training

In [ ]:
# ============================================================
#   🚀 START TRAINING JOB
#   This will take 5-15 minutes
#   Cost: ~$0.03
# ============================================================

print("="*60)
print("STARTING TRAINING JOB")
print("="*60)
print(f"")
print(f"⏱️  Expected time: 5-15 minutes")
print(f"💰 Estimated cost: ~$0.03")
print(f"")
print(f"What's happening:")
print(f"  1. Launching ml.m5.large instance...")
print(f"  2. Downloading XGBoost container...")
print(f"  3. Downloading training data from S3...")
print(f"  4. Training model (100 rounds)...")
print(f"  5. Saving model to S3...")
print(f"")
print(f"You can monitor progress in AWS Console:")
print(f"  SageMaker > Training > Training jobs")
print(f"")
print("-"*60)

# Start training!
xgb_estimator.fit(
    inputs={
        'train': train_input,
        'validation': val_input
    },
    wait=True,  # Wait for training to complete
    logs='All'  # Show all training logs
)

print("-"*60)
print("")
print("✅ TRAINING COMPLETE!")
print("")

---
## Cell 12: Verify Model Created

Confirm the model was saved to S3.

In [ ]:
# Get the model artifact location
model_artifact_uri = xgb_estimator.model_data

print("Model Training Results")
print("=" * 60)
print(f"Model artifact: {model_artifact_uri}")
print(f"")
print(f"This .tar.gz file contains:")
print(f"  - xgboost-model (binary model file)")
print(f"  - Training metadata")
print(f"")
print(f"✅ Model is ready for deployment!")

---
## Cell 13: 🌐 DEPLOY MODEL AS ENDPOINT

**What is an Endpoint?**  
A live web service that:
- Runs 24/7
- Accepts prediction requests
- Returns churn probability

**IMPORTANT:** Endpoints cost money ($0.05/hour) until deleted!

**Instance Type:**
- `ml.t2.medium` = 2 vCPU, 4 GB RAM
- Cheapest option: ~$0.05/hour

In [ ]:
# ============================================================
#   🌐 DEPLOY MODEL AS REAL-TIME ENDPOINT
#   This will take 5-8 minutes
#   Cost: ~$0.05/hour WHILE RUNNING
#   ⚠️  DELETE WHEN DONE TO STOP BILLING!
# ============================================================

print("="*60)
print("DEPLOYING ENDPOINT")
print("="*60)
print(f"")
print(f"⏱️  Expected time: 5-8 minutes")
print(f"💰 Cost: ~$0.05/hour while running")
print(f"")
print(f"⚠️  IMPORTANT: Delete endpoint when done testing!")
print(f"")
print("-"*60)

# Deploy!
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',  # Cheapest: ~$0.05/hour
    endpoint_name='churn-prediction-endpoint'
)

print("-"*60)
print(f"")
print(f"✅ ENDPOINT DEPLOYED!")
print(f"")
print(f"Endpoint name: churn-prediction-endpoint")
print(f"Status: InService")
print(f"")
print(f"⚠️  Remember to delete when done: Cell 17")

---
## Cell 14: Configure Predictor for Testing

Set up how to send data to the endpoint.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

# Configure how to send/receive data
predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

print("Predictor configured!")
print(f"Input format:  CSV")
print(f"Output format: CSV")

---
## Cell 15: 🧪 TEST PREDICTIONS

Send a sample customer to the endpoint and get churn probability.

In [ ]:
# Test with first customer from validation set
print("Testing Endpoint with Sample Customers")
print("=" * 60)

# Get a few test samples (without the target column)
# Remember: first column is target (Churn), rest are features
test_samples = val_df.iloc[:5]  # First 5 validation samples

for i, row in test_samples.iterrows():
    # Actual churn value (column 0)
    actual_churn = int(row.iloc[0])
    
    # Features (columns 1 onwards)
    features = row.iloc[1:].values
    features_str = ','.join([str(x) for x in features])
    
    # Get prediction from endpoint
    response = predictor.predict(features_str)
    
    # Parse probability (comes as [["0.1234"]])
    prob = float(response[0][0])
    
    # Determine prediction
    predicted_churn = 1 if prob > 0.5 else 0
    
    # Compare
    correct = "✓" if predicted_churn == actual_churn else "✗"
    actual_str = "CHURN" if actual_churn == 1 else "STAY"
    pred_str = "CHURN" if predicted_churn == 1 else "STAY"
    
    print(f"Customer {i+1}: Probability={prob:.1%} | Predicted={pred_str} | Actual={actual_str} | {correct}")

---
## Cell 16: Test Custom Customer

Create your own customer and get a prediction.

In [ ]:
# Test with a custom customer
# Use the feature order from your data (without the target column)

print("Custom Customer Prediction")
print("=" * 60)

# Get feature names (excluding target)
feature_names = list(df_prepared.columns[1:])
print(f"Features needed ({len(feature_names)}):")
for i, name in enumerate(feature_names[:10]):
    print(f"  {i+1}. {name}")
if len(feature_names) > 10:
    print(f"  ... and {len(feature_names) - 10} more")

print(f"\nUsing average values for a typical customer...")

# Use mean values for numeric features (a "typical" customer)
typical_customer = df_prepared.iloc[:, 1:].mean().values
typical_str = ','.join([str(round(x, 4)) for x in typical_customer])

# Get prediction
response = predictor.predict(typical_str)
prob = float(response[0][0])

print(f"\nPrediction for typical customer:")
print(f"  Churn probability: {prob:.1%}")
print(f"  Risk level: {'HIGH RISK' if prob > 0.5 else 'LOW RISK'}")

---
## Cell 17: ⚠️ DELETE ENDPOINT

**RUN THIS WHEN YOU'RE DONE TESTING!**

This stops the $0.05/hour billing.

In [ ]:
# ============================================================
#   ⚠️  DELETE ENDPOINT TO STOP BILLING
#   Run this when you're finished testing!
# ============================================================

print("Deleting endpoint...")
predictor.delete_endpoint()
print("")
print("✅ ENDPOINT DELETED!")
print("")
print("Billing has stopped.")
print("")
print("To redeploy later, run from Cell 13.")

---
## Summary

**What you accomplished:**
1. ✅ Loaded data from S3
2. ✅ Prepared data for XGBoost (numeric, target-first)
3. ✅ Trained XGBoost model on AWS infrastructure
4. ✅ Deployed model as real-time endpoint
5. ✅ Tested predictions
6. ✅ Cleaned up resources

**Cost for this session:**
- Training: ~$0.03
- Endpoint (assuming 1 hour): ~$0.05
- **Total: ~$0.08**

**Files created in S3:**
- `s3://customer-churn-vahant-2026/sagemaker/training-data/train.csv`
- `s3://customer-churn-vahant-2026/sagemaker/training-data/validation.csv`
- `s3://customer-churn-vahant-2026/sagemaker/models/churn-xgboost-.../output/model.tar.gz`